In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
def get_y_counts(df, digits=1, col="MT_Haplo"):
    """Get Y Chromosome counts from Dataframe df"""
    ys = df[col].str[:1]
    cts = ys.value_counts().values
    return cts
def get_y_counts_2(df, digits=2, col="MT_Haplo"):
    """Get Y Chromosome counts from Dataframe df"""
    ys = df[col].str[:2]
    cts = ys.value_counts().values
    return cts
def simpson_di(x):
    """ Given a count vector, returns the Simpson Diversity Index
    """
    n = np.sum(x) # Sample Size
    h = np.sum(x*(x-1)) / (n*(n-1)) # Fraction of pairs are identiclal
    
    if h==0: ### Set minimimum homo-cutoff (one homo-pair):
        h = 2 / (n*(n-1))
    return 1 / h

In [ ]:
Ymeta=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/Hap/mtHap/mtHap_120226.csv')

In [ ]:
import pandas as pd

def get_haplo_counts(df, col="MT_Haplo"):
    """
    Get haplogroup counts at first phylogenetic level:
    - 'HV' treated separately
    - otherwise first letter (H, V, J, K, T, U, etc.)
    Returns:
        counts array,
        value_counts series,
        total individuals
    """
    s = df[col].astype(str).str.strip()

    # classify haplogroups
    first_level = s.where(~s.str.startswith("HV"), "HV")
    first_level = first_level.where(first_level.eq("HV"), first_level.str[0])

    vc = first_level.value_counts()
    cts = vc.values
    total_n = len(first_level)

    return cts, vc, total_n


MT_LBA = Ymeta[Ymeta['Date'] == 'KNC_MLBA']
cts_LBA, vc_LBA, n_LBA = get_haplo_counts(MT_LBA)

MT_EIA = Ymeta[Ymeta['Date'] == 'KNC_EIA']
cts_EIA, vc_EIA, n_EIA = get_haplo_counts(MT_EIA)

MT_DIA = Ymeta[Ymeta['Date'] == 'KNC_DIA']
cts_DIA, vc_DIA, n_DIA = get_haplo_counts(MT_DIA)


#print("LBA counts:\n", vc_LBA)
print("Total LBA:", n_LBA)
print("Simpson LBA:", simpson_di(cts_LBA), "\n")

#print("EIA counts:\n", vc_EIA)
print("Total EIA:", n_EIA)
print("Simpson EIA:", simpson_di(cts_EIA), "\n")

#print("DIA counts:\n", vc_DIA)
print("Total DIA:", n_DIA)
print("Simpson DIA:", simpson_di(cts_DIA))



In [ ]:
def get_mt_counts(df, col="MT_Haplo"):
    """
    Count haplogroups at major clade level,
    keeping HV separate from H.
    """
    def extract_major(h):
        if pd.isna(h):
            return None
        h = str(h)
        if h.startswith("HV"):
            return "HV"
        else:
            return h[0]   # first letter for others

    major = df[col].apply(extract_major)
    cts = major.value_counts().values
    return cts

Ymeta_LBA = Ymeta[Ymeta['Date']=='KNC_MLBA']
cts_LBA = get_mt_counts(Ymeta_LBA)
Ymeta_EIA = Ymeta[Ymeta['Date']=='KNC_EIA']
cts_EIA = get_mt_counts(Ymeta_EIA)
Ymeta_IA = Ymeta[Ymeta['Date']=='KNC_DIA']
cts_IA = get_mt_counts(Ymeta_IA)
print(cts_LBA)
print(cts_EIA)
print(cts_IA)
SimInd_LBA = simpson_di(cts_LBA)
SimInd_EIA = simpson_di(cts_EIA)
SimInd_IA = simpson_di(cts_IA)
print(SimInd_LBA)
print(SimInd_EIA)
print(SimInd_IA)

In [ ]:
import numpy as np

def simpson_di(counts):
    """Calculate Simpson's Diversity Index from count array."""
    counts = np.array(counts)
    N = counts.sum()
    if N <= 1:
        return 0
    return 1 / (np.sum(counts * (counts - 1)) / (N * (N - 1)))

def bootstrap_simpson(counts, n_boot=10000):
    """Bootstrap distribution of Simpson index."""
    counts = np.array(counts)
    total = counts.sum()
    values = []

    for _ in range(n_boot):
        # Generate random sample with replacement
        sample = np.random.choice(np.repeat(np.arange(len(counts)), counts), size=total, replace=True)
        boot_counts = np.bincount(sample, minlength=len(counts))
        values.append(simpson_di(boot_counts))

    return np.array(values)

# Example usage


counts_EIA = np.array([50,23,13,11,10,9,6,2,1,1])
counts_DIA = np.array([31,18,15,7,5,4,3,2])   # Replace with actual counts

# Bootstrap both
boot_DIA = bootstrap_simpson(counts_DIA)
boot_EIA = bootstrap_simpson(counts_EIA)

# Calculate difference
diff = boot_DIA - boot_EIA
p_value = 2 * min(
    np.mean(diff >= 0),
    np.mean(diff <= 0)
)
  # two-tailed test

print(f"Mean DIA Simpson: {boot_DIA.mean():.3f}")
print(f"Mean EIA Simpson: {boot_EIA.mean():.3f}")
print(f"P-value for difference: {p_value:.4f}")
